# 01 — Data Acquisition

This notebook loads and inspects the four raw datasets, documents their structure and quirks, and produces clean CSV files in `data/processed/` for use in subsequent notebooks.

The core question this project is building toward: **do English local authorities with higher rates of SEN exclusions show higher rates of young people entering the youth justice system?** And does that relationship hold once we control for deprivation?

The four datasets:
1. **DfE exclusions by characteristic** — permanent exclusion rates by SEN status at LA level
2. **YJB first-time entrants** — rate of children entering the youth justice system per 100k, by LA
3. **IMD 2019** — deprivation index scores by LA, our control variable
4. **DfE SEN by school** — school-level SEN counts by primary need, aggregated to LA

---

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path('../data/raw')
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(exist_ok=True)

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

## 1. DfE Exclusions by Pupil Characteristic

Source: [Suspensions and permanent exclusions in England](https://explore-education-statistics.service.gov.uk/find-statistics/suspensions-and-permanent-exclusions-in-england) — download "by pupil characteristic".

This covers academic years 2019/20 through 2024/25. It includes permanent exclusion rates broken down by SEN provision status (EHC plan, SEN support, no identified SEN) at local authority level. 587,925 rows total across all geographies, characteristics and years — we filter to SEN provision at LA level.

In [2]:
exc_raw = pd.read_csv(RAW / 'exc_characteristics.csv')
print(f'Shape: {exc_raw.shape}')
print(f'Years: {sorted(exc_raw["time_period"].unique())}')
print(f'Geographic levels: {exc_raw["geographic_level"].unique()}')
print(f'Characteristic groups: {exc_raw["characteristic_group"].unique()}')

Shape: (587925, 20)
Years: [201920, 202021, 202122, 202223, 202324, 202425]
Geographic levels: ['National' 'Regional' 'Local authority']
Characteristic groups: ['Age' 'Ethnicity Major' 'Ethnicity Minor' 'FSM' 'SEN provision' 'Sex'
 'Total' 'Year group']


In [3]:
# Filter to SEN provision, LA level, Total education phase
exc_la = exc_raw[
    (exc_raw['characteristic_group'] == 'SEN provision') &
    (exc_raw['geographic_level'] == 'Local authority') &
    (exc_raw['education_phase'] == 'Total')
][['new_la_code', 'la_name', 'time_period', 'characteristic',
   'headcount', 'perm_excl', 'perm_excl_rate', 'susp_rate']].copy()

exc_la['perm_excl_rate'] = pd.to_numeric(exc_la['perm_excl_rate'], errors='coerce')
exc_la['susp_rate'] = pd.to_numeric(exc_la['susp_rate'], errors='coerce')

print(f'Shape: {exc_la.shape}')
print(f'Unique LAs: {exc_la["la_name"].nunique()}')
print(f'SEN categories: {exc_la["characteristic"].unique()}')
print(f'Suppressed perm_excl_rate values: {exc_la["perm_excl_rate"].isna().sum()}')
exc_la.head(8)

Shape: (7748, 8)
Unique LAs: 155
SEN categories: ['EHC plan' 'No identified SEN' 'SEN support' 'Unclassified']
Suppressed perm_excl_rate values: 0


,new_la_code,la_name,time_period,characteristic,headcount,perm_excl,perm_excl_rate,susp_rate
2352,E06000001,Hartlepool,202425,EHC plan,647,2,0.30912,21.79289
2353,E06000001,Hartlepool,202425,No identified SEN,11739,6,0.05111,8.58676
2354,E06000001,Hartlepool,202425,SEN support,2445,8,0.32720,26.33947
2555,E06000002,Middlesbrough,202425,EHC plan,1460,0,0.00000,11.16438
2556,E06000002,Middlesbrough,202425,No identified SEN,20036,9,0.04492,5.77461
2557,E06000002,Middlesbrough,202425,SEN support,3868,9,0.23268,19.70010
2760,E06000003,Redcar and Cleveland,202425,EHC plan,988,2,0.20243,15.08097
2761,E06000003,Redcar and Cleveland,202425,No identified SEN,16246,6,0.03693,8.04506


In [4]:
exc_la.to_csv(PROCESSED / 'exclusions_sen_la.csv', index=False)
print('Saved: exclusions_sen_la.csv')

Saved: exclusions_sen_la.csv


## 2. YJB First-Time Entrants by Local Authority

Source: [Youth Justice Statistics](https://www.gov.uk/government/statistics/youth-justice-statistics) — Chapter 2 Excel file.

Sheet 2.11 gives rates per 100,000 of the 10–17 population — the right measure for comparing across LAs of different sizes, and it means we don't need to fetch separate ONS population data.

**Temporal note:** YJB data runs to calendar year-end (December). DfE exclusions data runs to academic year-end (August). YJB 'year ending December 2023' is the closest match to DfE academic year 2022/23. This ~4 month offset is noted and applied consistently throughout the analysis.

In [7]:
yjb_raw = pd.read_excel(
    RAW / 'Ch_2_-_First_time_entrants_to_the_youth_justice_system.xlsx',
    sheet_name='2.11',
    header=3  # row 3 is the actual header; rows 0-2 are title/notes
)

yjb_raw = yjb_raw.rename(columns={
    'Region': 'region',
    'Local Authority/Region': 'la_name'
})

print(f'Shape: {yjb_raw.shape}')
print(f'Regions: {yjb_raw["region"].unique()}')

Shape: (191, 13)
Regions: ['Wales' 'East Midlands' 'Eastern' 'London' 'North East' 'North West'
 'South East' 'South West' 'West Midlands' 'Yorkshire and the Humber'
 'England and Wales']


In [8]:
# Keep English LAs only — drop Wales and regional subtotals
english_regions = [
    'East Midlands', 'East of England', 'London', 'North East',
    'North West', 'South East', 'South West', 'West Midlands',
    'Yorkshire and the Humber'
]
yjb_eng = yjb_raw[yjb_raw['region'].isin(english_regions)].copy()
print(f'English LAs: {yjb_eng["la_name"].nunique()}')

# Melt from wide (one col per year) to long (one row per LA per year)
year_cols = [c for c in yjb_eng.columns if str(c).strip() not in ['region', 'la_name']]
yjb_long = yjb_eng.melt(
    id_vars=['region', 'la_name'],
    value_vars=year_cols,
    var_name='yjb_year',
    value_name='fte_rate_per_100k'
)
yjb_long['yjb_year'] = pd.to_numeric(yjb_long['yjb_year'], errors='coerce').astype('Int64')
yjb_long['fte_rate_per_100k'] = pd.to_numeric(yjb_long['fte_rate_per_100k'], errors='coerce')

print(f'Shape after melt: {yjb_long.shape}')
print(f'Years covered: {sorted(yjb_long["yjb_year"].dropna().unique())}')
print(f'Missing FTE rates: {yjb_long["fte_rate_per_100k"].isna().sum()} (suppressed small-count LAs)')
yjb_long.head()

English LAs: 154
Shape after melt: (1705, 4)
Years covered: [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Missing FTE rates: 106 (suppressed small-count LAs)


,region,la_name,yjb_year,fte_rate_per_100k
0,East Midlands,Derby,2014,543.958
1,East Midlands,Derbyshire,2014,275.226
2,East Midlands,Leicester,2014,578.026
3,East Midlands,Leicestershire,2014,314.159
4,East Midlands,Lincolnshire,2014,406.993


In [9]:
yjb_long.to_csv(PROCESSED / 'yjb_fte_la.csv', index=False)
print('Saved: yjb_fte_la.csv')

Saved: yjb_fte_la.csv


## 3. IMD 2019 — Deprivation Index

Source: [English Indices of Deprivation 2019](https://www.gov.uk/government/statistics/english-indices-of-deprivation-2019) — File 10, lower-tier LA summaries.

We use `IMD - Average score` per LA as the deprivation control variable. Higher score = more deprived.

The ONS LA code (`Local Authority District code (2019)`) matches `new_la_code` in the DfE data directly — no name-matching required for this join.

In [10]:
imd_raw = pd.read_excel(
    RAW / 'File_10_-_IoD2019_Local_Authority_District_Summaries__lower-tier__.xlsx',
    sheet_name='IMD'
)

imd = imd_raw[[
    'Local Authority District code (2019)',
    'Local Authority District name (2019)',
    'IMD - Average score '
]].copy()
imd.columns = ['la_code', 'la_name_imd', 'imd_avg_score']

print(f'Shape: {imd.shape}')
print(f'Missing scores: {imd["imd_avg_score"].isna().sum()}')
print(f'Score range: {imd["imd_avg_score"].min():.1f} – {imd["imd_avg_score"].max():.1f}')
imd.head()

Shape: (317, 3)
Missing scores: 0
Score range: 5.5 – 45.0


,la_code,la_name_imd,imd_avg_score
0,E06000001,Hartlepool,35.037
1,E06000002,Middlesbrough,40.460
2,E06000003,Redcar and Cleveland,29.792
3,E06000004,Stockton-on-Tees,25.790
4,E06000005,Darlington,25.657


In [11]:
imd.to_csv(PROCESSED / 'imd_la.csv', index=False)
print('Saved: imd_la.csv')

Saved: imd_la.csv


## 4. DfE SEN by Primary Need — Aggregated to LA Level

Source: [Special educational needs in England 2024/25](https://explore-education-statistics.service.gov.uk/find-statistics/special-educational-needs-in-england/2024-25) — school-level underlying data file (`sen_school_level_ud.csv`).

The DfE no longer publishes a pre-aggregated LA-level SEN needs breakdown in the standard download (from 2025/26 it moves to API-only). The school-level file contains every school in England with counts for each primary need type, broken down by EHC plan and SEN support separately. We aggregate to LA level by summing.

This gives us the SEN population profile of each LA — what proportion of SEN children have SEMH (Social, Emotional and Mental Health), SPLD (Specific Learning Difficulties: dyslexia, ADHD), ASD, etc. Used in the secondary analysis to ask whether the exclusion-to-justice pathway is stronger in LAs with higher prevalence of specific need types.

**Limitation:** single year snapshot (2024/25) rather than a time series.

In [12]:
school_raw = pd.read_csv(RAW / 'sen_school_level_ud.csv', low_memory=False)
print(f'Shape: {school_raw.shape}')
print(f'Unique LAs: {school_raw["new_la_code"].nunique()}')
print(f'Year: {school_raw["time_period"].unique()}')

Shape: (24479, 75)
Unique LAs: 153
Year: [202425]


In [13]:
need_cols = [
    'Total pupils', 'SEN support', 'EHC plan',
    'EHC_Primary_need_spld', 'EHC_Primary_need_mld',
    'EHC_Primary_need_semh', 'EHC_Primary_need_slcn', 'EHC_Primary_need_asd',
    'SUP_Primary_need_spld', 'SUP_Primary_need_mld',
    'SUP_Primary_need_semh', 'SUP_Primary_need_slcn', 'SUP_Primary_need_asd',
]

sen_la = school_raw.groupby(['new_la_code', 'la_name'])[need_cols].sum().reset_index()

# Combined totals across EHC + support
sen_la['total_sen']  = sen_la['SEN support'] + sen_la['EHC plan']
sen_la['total_semh'] = sen_la['EHC_Primary_need_semh'] + sen_la['SUP_Primary_need_semh']
sen_la['total_spld'] = sen_la['EHC_Primary_need_spld'] + sen_la['SUP_Primary_need_spld']
sen_la['total_asd']  = sen_la['EHC_Primary_need_asd']  + sen_la['SUP_Primary_need_asd']
sen_la['total_slcn'] = sen_la['EHC_Primary_need_slcn'] + sen_la['SUP_Primary_need_slcn']

# Rates as % of all SEN pupils in the LA
sen_la['pct_semh'] = (sen_la['total_semh'] / sen_la['total_sen'] * 100).round(2)
sen_la['pct_spld'] = (sen_la['total_spld'] / sen_la['total_sen'] * 100).round(2)
sen_la['pct_asd']  = (sen_la['total_asd']  / sen_la['total_sen'] * 100).round(2)
sen_la['pct_slcn'] = (sen_la['total_slcn'] / sen_la['total_sen'] * 100).round(2)

print(f'Shape: {sen_la.shape}')
sen_la[['la_name', 'total_sen', 'pct_semh', 'pct_spld', 'pct_asd']].head(8)

Shape: (153, 24)


,la_name,total_sen,pct_semh,pct_spld,pct_asd
0,Hartlepool,3300,16.88,9.42,10.97
1,Middlesbrough,5685,21.50,9.27,10.64
2,Redcar and Cleveland,4515,24.27,12.93,12.27
3,Stockton-on-Tees,6551,16.27,7.27,10.65
4,Darlington,3417,20.98,8.66,17.82
5,Halton,4285,24.71,8.73,9.75
6,Warrington,6256,23.80,9.61,11.84
7,Blackburn with Darwen,5775,22.55,7.32,9.54


In [14]:
sen_la.to_csv(PROCESSED / 'sen_needs_la.csv', index=False)
print('Saved: sen_needs_la.csv')

Saved: sen_needs_la.csv


## Summary

Four processed files written to `data/processed/`:

| File | Shape | Join key | Notes |
|---|---|---|---|
| `exclusions_sen_la.csv` | 7,748 × 8 | `new_la_code` | 155 LAs, 6 years, 4 SEN categories |
| `yjb_fte_la.csv` | 1,705 × 4 | `la_name` (needs code lookup) | 154 English LAs, 2014–2024 |
| `imd_la.csv` | 317 × 3 | `la_code` | ONS code matches DfE `new_la_code` directly |
| `sen_needs_la.csv` | 153 × 23 | `new_la_code` | 2024/25 snapshot only |

**Key issue for notebook 02:** the YJB data has no LA codes, only names. We need to build a name→code lookup to join it to the other three datasets. Notebook 02 handles this, along with aligning the temporal offset between DfE academic years and YJB calendar years.